# PUDM — Environment Setup

Run this notebook **once** (or after a PyTorch/CUDA version change) to:
1. Clone the repo to Google Drive (persistent across sessions)
2. Install Python dependencies
3. Compile CUDA extensions and cache them inside the repo

After setup, the training notebooks (`pudm_ddpm.ipynb`, `pudm_flow_matching.ipynb`) can be run directly — they assume this setup is done.

### Data

Download and unzip datasets into `data/` inside the repo on Drive:
```
pudm_extension/
  data/
    PU1K/
      train/pu1k_poisson_256_poisson_1024_pc_2500_patch50_addpugan.h5
      test/...
    PUGAN/
      train/PUGAN_poisson_256_poisson_1024.h5
      test/...
```

**Requirements:** Colab GPU runtime (T4 or better).

In [ ]:
import os
import subprocess

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone (or update) the repo on Drive
REPO_DIR = '/content/drive/MyDrive/MVA/pointcloud/pudm_extension'
GITHUB_URL = 'https://github.com/valbad/pudm_extension.git'

if not os.path.exists(REPO_DIR):
    print('Cloning repo to Drive (first time)...')
    result = subprocess.run(
        ['git', 'clone', GITHUB_URL, REPO_DIR],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f'git clone failed:\n{result.stderr}')
    print(f'Cloned to {REPO_DIR}')
else:
    print(f'Repo exists at {REPO_DIR} — pulling latest...')
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull', '--ff-only'],
        capture_output=True, text=True,
    )
    print(result.stdout.strip() or 'Already up to date.')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# torch is pre-installed on Colab; ninja speeds up JIT compilation
!pip install -q ninja

import torch, sys
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')
print(f'GPU:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Compile CUDA extensions (cached in repo/cache/ on Drive)
import shutil, glob as _glob, time as _time

# Tag by Python + PyTorch + CUDA versions
_tag = f'py{sys.version_info.major}{sys.version_info.minor}_pt{torch.__version__.split("+")[0]}_cu{torch.version.cuda.replace(".", "")}'
CACHE_DIR = os.path.join(REPO_DIR, 'cache', _tag)
os.makedirs(CACHE_DIR, exist_ok=True)

POINTOPS_SRC = os.path.join(REPO_DIR, 'src', 'ops', 'pointops')
PNET2_PKG    = os.path.join(REPO_DIR, 'src', 'ops', 'pointnet2_ops')

# ---- pointops ----
pointops_cache = os.path.join(CACHE_DIR, 'pointops')
cached_sos = _glob.glob(os.path.join(pointops_cache, 'pointops_cuda*.so'))
if cached_sos:
    for so in cached_sos:
        shutil.copy2(so, POINTOPS_SRC)
    print(f'\u2713 pointops: restored from cache')
else:
    print('pointops: compiling...')
    os.environ.pop('TORCH_CUDA_ARCH_LIST', None)
    %cd {POINTOPS_SRC}
    !python setup.py build_ext --inplace 2>&1 | tail -5
    %cd {REPO_DIR}
    built = _glob.glob(os.path.join(POINTOPS_SRC, 'pointops_cuda*.so'))
    if built:
        os.makedirs(pointops_cache, exist_ok=True)
        for so in built:
            shutil.copy2(so, pointops_cache)
        print(f'\u2713 pointops: compiled and cached')
    else:
        print('\u26a0 pointops: build failed')

# ---- pointnet2_ops ----
pnet2_cache = os.path.join(CACHE_DIR, 'pointnet2_ops')
cached_ext = _glob.glob(os.path.join(pnet2_cache, '_ext*.so'))
if cached_ext:
    for so in cached_ext:
        shutil.copy2(so, PNET2_PKG)
    print(f'\u2713 pointnet2_ops: restored from cache')
else:
    print('pointnet2_ops: JIT compiling...')
    sys.path.insert(0, REPO_DIR)
    t0 = _time.time()
    from src.ops.pointnet2_ops import pointnet2_utils  # triggers JIT
    dt = _time.time() - t0
    # Find .so in JIT cache and copy to our cache
    _jit_sos = _glob.glob(os.path.expanduser('~/.cache/torch_extensions/**/_ext.so'), recursive=True)
    if _jit_sos:
        os.makedirs(pnet2_cache, exist_ok=True)
        shutil.copy2(_jit_sos[0], pnet2_cache)
        shutil.copy2(_jit_sos[0], PNET2_PKG)
        print(f'\u2713 pointnet2_ops: compiled in {dt:.0f}s and cached')
    else:
        print(f'\u2713 pointnet2_ops: compiled in {dt:.0f}s (could not locate .so to cache)')

print(f'\nAll CUDA extensions ready  (env: {_tag})')
print(f'Cache location: {CACHE_DIR}')

In [ ]:
# Verify everything imports correctly
sys.path.insert(0, REPO_DIR)

from src.generative import get_strategy, STRATEGIES
from src.utils.config import load_config
from src.utils.misc import set_seed

print(f'Strategies: {list(STRATEGIES.keys())}')
print('All imports OK!')

In [ ]:
# Check data directories
DATA_DIR = os.path.join(REPO_DIR, 'data')
for ds in ['PU1K', 'PUGAN']:
    ds_dir = os.path.join(DATA_DIR, ds)
    if os.path.isdir(ds_dir):
        n_files = sum(len(f) for _, _, f in os.walk(ds_dir))
        print(f'\u2713 {ds}: {n_files} files in {ds_dir}')
    else:
        print(f'\u2298 {ds}: not found — download and unzip into {ds_dir}')

## Done!

You can now open `pudm_ddpm.ipynb` or `pudm_flow_matching.ipynb` and run them directly.

```
pudm_extension/          ← on Google Drive
├── cache/               ← compiled CUDA .so files (auto-populated)
├── data/                ← datasets (you populate this)
│   ├── PU1K/
│   └── PUGAN/
├── configs/
├── notebooks/
│   ├── config.ipynb     ← this notebook (run once)
│   ├── pudm_ddpm.ipynb
│   └── pudm_flow_matching.ipynb
└── src/
```